# face-cloak live demo backend (Kaggle GPU)

> ## WARNING - keep this notebook private
> **Never make this notebook, or any version of it, public. Never use "Save Version" with outputs** (or "Save & Run All"
> commits): cell outputs contain the live tunnel URL, and Kaggle versions/outputs can be shared or indexed.
> The passcode is read from **Kaggle Secrets** and is never printed (only a masked form like `ab***yz`).
> Before sharing or committing this file run `nbstripout` (see `server/README.md`) so the saved copy has no outputs.

Runs the FastAPI backend from `server/` on a Kaggle GPU and exposes it through a free Cloudflare quick tunnel,
so the website at https://face-cloak.vercel.app/demo can call it from the browser.

**Before you run**
1. Right panel -> **Settings -> Accelerator: GPU T4 x2 (or P100)**, **Internet: ON**.
2. Choose a passcode of **at least 12 characters**: **Add-ons -> Secrets -> add secret `DEMO_KEY`** (required).
   Generate one with `python -c "import secrets; print(secrets.token_urlsafe(12))"`.
3. **Run All**. Open the big link printed by cell 6. Keep this tab open: the last cell is the keep-alive.

The tunnel URL changes on every run. The backend keeps jobs in memory for 10 minutes only and writes no images to disk.
Logs and the cloudflared binary live in `/tmp`, not in `/kaggle/working` (which is what "Save Version" captures).

Only upload photos of yourself or people who consented. Images are processed in memory and deleted within 10 minutes.

In [ ]:
# 1) Config. DEMO_KEY comes ONLY from the Kaggle Secret `DEMO_KEY` (never typed into the notebook, never printed in full)
import os
DEMO_SITE = "https://face-cloak.vercel.app"
PORT = 8000
# optional: your Vercel team slug, to allow CORS from your preview deployments
# os.environ["VERCEL_TEAM_SLUG"] = "your-team"
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("DEMO_KEY")
except Exception as e:
    raise SystemExit(f"No Kaggle secret DEMO_KEY ({type(e).__name__}). Add it under Add-ons -> Secrets (>=12 chars).")
if len(key) < 12:
    raise SystemExit("DEMO_KEY must be at least 12 characters.")
os.environ["DEMO_KEY"] = key
os.environ["DEMO_SITE"] = DEMO_SITE
os.environ["FACECLOAK_THREADS"] = "4"
print(f"Passcode loaded from Kaggle Secrets: {key[:2]}***{key[-2:]}  (see Kaggle Secrets for the full value)")
del key

In [ ]:
# 2) Clone the repo and install dependencies (torch/CUDA are preinstalled on Kaggle: do NOT reinstall torch)
# REPO_REF: set this to a release tag or commit SHA for the real demo so the code cannot change under you.
REPO_REF = "main"
%cd /kaggle/working
!rm -rf face-cloak
!git clone -q https://github.com/J4KE-B/face-cloak
!git -C face-cloak checkout -q {REPO_REF}
%cd /kaggle/working/face-cloak
!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -e .
# floors = the versions in pyproject.toml / uv.lock (python-multipart and pillow have had parser/decoder CVEs)
!pip install -q -U "python-multipart>=0.0.32" "pillow>=12.3.0" "fastapi>=0.143.0" "uvicorn>=0.54.0"

In [ ]:
# 3) Model weights (inswapper_128, buffalo_l, antelopev2 judge, facenet; SimSwap is not needed for the demo)
!python -m facecloak.weights --fetch

In [ ]:
# 4) Start the backend in the background; logs go to /tmp (short job-id hashes and timings only, no images)
import subprocess, time, urllib.request, json, sys
LOG = "/tmp/backend.log"
open(LOG, "w").close(); os.chmod(LOG, 0o600)
backend = subprocess.Popen(
    [sys.executable, "-m", "server", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/kaggle/working/face-cloak", env=os.environ.copy(),
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
)
health = None
for _ in range(180):
    if backend.poll() is not None:
        print(open(LOG).read()[-3000:]); raise SystemExit("backend exited")
    try:
        health = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3)); break
    except Exception:
        time.sleep(2)
else:
    print(open(LOG).read()[-3000:]); raise SystemExit("backend did not become healthy in 6 min")
print("backend healthy:", health)

In [ ]:
# 5) Cloudflare quick tunnel: pinned official cloudflared release, sha256-verified, stored in /tmp
import re, stat, pathlib, hashlib
CF_VERSION = "2026.10.0"
CF_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"  # from the GitHub release notes
cf = pathlib.Path(f"/tmp/cloudflared-{CF_VERSION}")
if not cf.exists():
    part = cf.with_name(cf.name + ".part")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64", part)
    if hashlib.sha256(part.read_bytes()).hexdigest() != CF_SHA256:
        part.unlink(); raise SystemExit("cloudflared checksum mismatch - refusing to run it")
    part.chmod(part.stat().st_mode | stat.S_IEXEC); part.rename(cf)
TLOG = "/tmp/tunnel.log"
tunnel = subprocess.Popen([str(cf), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                          stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
tunnel_url = None
for _ in range(90):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TLOG).read())
    if m: tunnel_url = m.group(0); break
    if tunnel.poll() is not None: print(open(TLOG).read()[-2000:]); raise SystemExit("cloudflared exited")
    time.sleep(1)
assert tunnel_url, "no tunnel URL found; see " + TLOG
print("tunnel:", tunnel_url)

In [ ]:
# 6) The link to open (the passcode is NOT shown: read it from your Kaggle Secret `DEMO_KEY`)
import urllib.parse
from IPython.display import HTML, display
link = f"{DEMO_SITE}/demo?backend={urllib.parse.quote(tunnel_url, safe='')}"
masked = f"{os.environ['DEMO_KEY'][:2]}***{os.environ['DEMO_KEY'][-2:]}"
display(HTML(f'''
<div style="padding:18px;border:2px solid #4f46e5;border-radius:12px;font-family:sans-serif">
  <div style="font-size:20px;margin-bottom:8px"><b>Demo backend is live</b></div>
  <a href="{link}" target="_blank" style="font-size:26px;font-weight:700">Open the demo (auto-connect)</a>
  <div style="margin-top:12px">Backend URL: <code>{tunnel_url}</code></div>
  <div>Passcode: <code>{masked}</code> - the full value is your Kaggle Secret <code>DEMO_KEY</code></div>
  <div style="margin-top:8px;color:#b91c1c"><b>Do not save a version with these outputs or make this notebook public.</b></div>
</div>'''))

In [ ]:
# 7) Keep-alive: prints health every 60 s. Interrupt the kernel to stop. (Logs are not echoed here on purpose.)
import datetime
try:
    while True:
        try:
            h = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5))
            line = f"queue={h['queue_length']} device={h['device']}"
        except Exception as e:
            line = f"HEALTH FAILED: {type(e).__name__}"
        print(datetime.datetime.now().strftime("%H:%M:%S"), line)
        time.sleep(60)
except KeyboardInterrupt:
    backend.terminate(); tunnel.terminate(); print("stopped")